# 04 — Revenue Gap Map

Maps the **total annual property tax revenue gap** by county — the estimated additional tax revenue
that would be collected if all residential properties were assessed at current market value
instead of their Prop 13-limited assessed values.

Formula: `Tax Gap ($) = (Market Value − Assessed Value) × 1.1%`

The 1.1% rate approximates California's 1% base rate plus average local special assessments/bonds.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'county_prop13.csv')
df['fips'] = df['fips'].astype(str).str.zfill(5)
df = df.dropna(subset=['tax_gap_annual_millions'])

total_gap = df['tax_gap_annual_millions'].sum()
print(f'Statewide annual residential property tax gap: ${total_gap:,.0f}M')
print(f'That is ${total_gap/1000:,.1f}B per year statewide')

Statewide annual residential property tax gap: $11,149M
That is $11.1B per year statewide


In [2]:
# Choropleth: total annual tax gap by county
fig = px.choropleth(
    df,
    geojson='https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json',
    locations='fips',
    color='tax_gap_annual_millions',
    color_continuous_scale='YlOrRd',
    scope='usa',
    hover_name='county',
    hover_data={
        'tax_gap_annual_millions': ':$,.0f',
        'tax_gap_per_household': ':$,.0f',
        'assessment_ratio': ':.1%',
        'fips': False,
    },
    labels={
        'tax_gap_annual_millions': 'Annual Gap ($M)',
        'tax_gap_per_household': 'Gap per HH ($)',
        'assessment_ratio': 'Assessment Ratio',
    },
    title='Annual Property Tax Revenue Gap by CA County ($M)<br>'
          '<sup>Estimated additional revenue if all homes assessed at market value</sup>',
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(
    margin={'l': 0, 'r': 0, 't': 60, 'b': 0},
    coloraxis_colorbar=dict(
        title='Annual Gap<br>($M)',
        tickprefix='$',
        ticksuffix='M',
    ),
    height=600,
)
fig.show()

In [3]:
# Choropleth: tax gap per household
fig2 = px.choropleth(
    df,
    geojson='https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json',
    locations='fips',
    color='tax_gap_per_household',
    color_continuous_scale='PuRd',
    scope='usa',
    hover_name='county',
    hover_data={
        'tax_gap_per_household': ':$,.0f',
        'assessment_ratio': ':.1%',
        'fips': False,
    },
    labels={
        'tax_gap_per_household': 'Annual Gap per HH ($)',
        'assessment_ratio': 'Assessment Ratio',
    },
    title='Annual Property Tax Gap per Owner-Occupied Household by County<br>'
          '<sup>Individual household benefit from Prop 13 assessment limits</sup>',
)
fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    margin={'l': 0, 'r': 0, 't': 60, 'b': 0},
    coloraxis_colorbar=dict(
        title='Annual Gap<br>per HH',
        tickprefix='$',
        tickformat=',.0f',
    ),
    height=600,
)
fig2.show()

In [4]:
# Summary table: top 15 counties by total gap
summary = df.nlargest(15, 'tax_gap_annual_millions')[[
    'county', 'assessment_ratio', 'market_value_millions',
    'boe_residential_av_millions', 'tax_gap_annual_millions', 'tax_gap_per_household'
]].copy()
summary.columns = [
    'County', 'Assessment Ratio', 'Market Value ($M)',
    'Assessed Value ($M)', 'Annual Tax Gap ($M)', 'Gap per HH ($)'
]
summary.style.format({
    'Assessment Ratio': '{:.1%}',
    'Market Value ($M)': '${:,.0f}M',
    'Assessed Value ($M)': '${:,.0f}M',
    'Annual Tax Gap ($M)': '${:,.0f}M',
    'Gap per HH ($)': '${:,.0f}',
})

,County,Assessment Ratio,Market Value ($M),Assessed Value ($M),Annual Tax Gap ($M),Gap per HH ($)
29,Orange,73.4%,"$705,307M","$517,680M","$2,064M","$3,423"
42,Santa Clara,73.3%,"$595,555M","$436,480M","$1,750M","$4,840"
18,Los Angeles,89.6%,"$1,364,121M","$1,221,960M","$1,564M","$1,007"
32,Riverside,66.2%,"$309,943M","$205,140M","$1,153M","$2,244"
0,Alameda,74.9%,"$338,194M","$253,440M",$932M,"$2,937"
35,San Bernardino,69.2%,"$220,560M","$152,640M",$747M,"$1,853"
33,Sacramento,81.2%,"$171,924M","$139,680M",$355M,"$1,079"
40,San Mateo,88.2%,"$246,555M","$217,440M",$320M,"$2,035"
6,Contra Costa,86.4%,"$213,193M","$184,260M",$318M,"$1,160"
20,Marin,73.3%,"$94,981M","$69,600M",$279M,"$4,202"
